### Align Market Reactions with Fed Announcements

In [17]:
# import necessary libraries
import pandas as pd
import datetime

In [18]:
# read csvs into respective dfs
statements = pd.read_csv("../../data/raw/fomc_statements.csv")
minutes = pd.read_csv("../../data/raw/fomc_minutes.csv")
speeches = pd.read_csv("../../data/raw/fed_speeches.csv")

statements["event_type"] = "statement"
minutes["event_type"] = "minutes"
speeches["event_type"] = "speech"

In [19]:
print(statements.columns)
print(minutes.columns)
print(speeches.columns)

Index(['date', 'meeting', 'url', 'text', 'event_type'], dtype='str')
Index(['date', 'meeting', 'url', 'text', 'event_type'], dtype='str')
Index(['date', 'title', 'speaker', 'url', 'text', 'event_type'], dtype='str')


In [20]:
# concatenate the dfs into a single one
events = pd.concat([statements, minutes, speeches])
events.shape

(418, 7)

In [21]:
# date column seems to have different data
print(events['date'].head(5))
events["date"].tail(5)

0    2010-01-27
1    2010-03-16
2    2010-04-28
3    2010-05-09
4    2010-06-23
Name: date, dtype: str


100    2/22/2017 1:00:00 PM
101    1/19/2017 8:00:00 PM
102    1/18/2017 3:00:00 PM
103    1/12/2017 7:00:00 PM
104    1/7/2017 11:15:00 AM
Name: date, dtype: str

In [22]:
# convert the date to datetime and sort the df by the date of different fed events
events["date"] = pd.to_datetime(events["date"], format="mixed")
events["date"] = events["date"].dt.normalize()
events = events.sort_values("date").reset_index(drop=True)

In [23]:


# Define your bounds
start_date = '2008-01-01'
end_date = '2025-12-31'

# Create the mask
mask = (events['date'] >= start_date) & (events['date'] <= end_date)

# Apply the filter
events = events.loc[mask].copy()

# Check the results
print(f"Filtered shape: {events.shape}")

Filtered shape: (416, 7)


In [24]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 416 entries, 0 to 415
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        416 non-null    datetime64[us]
 1   meeting     312 non-null    str           
 2   url         416 non-null    str           
 3   text        255 non-null    object        
 4   event_type  416 non-null    str           
 5   title       104 non-null    str           
 6   speaker     104 non-null    str           
dtypes: datetime64[us](1), object(1), str(5)
memory usage: 22.9+ KB


In [25]:
# load the market data
market = pd.read_csv("../../data/raw/market_data.csv")
market["date"] = pd.to_datetime(market["Date"])
market = market.sort_values("date").reset_index(drop=True)
market.head(5)

,Date,sp500_open,sp500_high,sp500_low,sp500_close,sp500_volume,vix,treasury_10y,treasury_2y,sp500_log_return,realized_volatility,date
0,2008-01-02,1467.969971,1471.770020,1442.069946,1447.160034,3452650000,23.170000,3.901,3.170,NaN,NaN,2008-01-02
1,2008-01-03,1447.550049,1456.800049,1443.729980,1447.160034,3429500000,22.490000,3.901,3.150,0.000000,NaN,2008-01-03
2,2008-01-04,1444.010010,1444.010010,1411.189941,1411.630005,4166000000,23.940001,3.854,3.115,-0.024858,NaN,2008-01-04
3,2008-01-07,1414.069946,1423.869995,1403.449951,1416.180054,4221260000,23.790001,3.839,3.165,0.003218,NaN,2008-01-07
4,2008-01-08,1415.709961,1430.280029,1388.300049,1390.189941,4705390000,25.430000,3.840,3.160,-0.018523,NaN,2008-01-08


In [ ]:
# align the fed event to a trading day and if it occurs on a day where market is closed, align it to the next trading day
def align_to_trading_day(event_date, market_dates):

    if event_date in market_dates.values:
        return event_date

    future_dates = market_dates[market_dates > event_date]

    if len(future_dates) == 0:
        return pd.NaT

    return future_dates.iloc[0]

events["aligned_date"] = events["date"].apply(
    lambda d: align_to_trading_day(d, market["date"])
)
events = events.dropna(subset=["aligned_date"])

events = events[
    (events["aligned_date"] >= market["date"].iloc[65]) &
    (events["aligned_date"] <= market["date"].iloc[-31])
]

In [27]:
# create event windows:
# estimation window: t-65 days to t-5 days for normal market behaviour
# event window: t=0 to t+30 days for changes due to fed events
def extract_event_window(event_date, market):

    idx = market.index[market["date"] == event_date][0]

    estimation_window = market.iloc[idx-65: idx-5]
    event_window = market.iloc[idx: idx+31]

    return estimation_window, event_window

In [28]:
# generate a df for all the events
event_windows = []

for i, row in events.iterrows():

    event_date = row["aligned_date"]

    try:
        est_win, evt_win = extract_event_window(event_date, market)

        evt_win = evt_win.copy()

        evt_win["t"] = range(0, len(evt_win))
        evt_win["event_id"] = i
        evt_win["event_date"] = event_date

        event_windows.append(evt_win)

    except:
        continue

event_windows_df = pd.concat(event_windows)

In [29]:
# a df for estimation 
estimation_windows = []

for i, row in events.iterrows():

    event_date = row["aligned_date"]

    try:

        est_win, evt_win = extract_event_window(event_date, market)

        est_win = est_win.copy()

        est_win["event_id"] = i
        est_win["event_date"] = event_date

        estimation_windows.append(est_win)

    except:
        continue

estimation_df = pd.concat(estimation_windows)

In [30]:
# forward fill the treasury data since it doesnt change value directly
market["treasury_10y"] = market["treasury_10y"].ffill()
market["treasury_2y"] = market["treasury_2y"].ffill()

In [31]:
events.tail(5)

,date,meeting,url,text,event_type,title,speaker,aligned_date
411,2025-10-29,"October 28-29, 2025",https://www.federalreserve.gov/monetarypolicy/...,NaN,minutes,NaN,NaN,2025-10-29
412,2025-10-29,"October 28-29, 2025",https://www.federalreserve.gov/newsevents/pres...,Available indicators suggest that economic act...,statement,NaN,NaN,2025-10-29
413,2025-12-01,NaN,https://www.federalreserve.gov/newsevents/spee...,I am deeply honored to have been asked to spea...,speech,Opening Remarks,Chair Jerome H. Powell,2025-12-01
414,2025-12-10,"December 9-10, 2025",https://www.federalreserve.gov/newsevents/pres...,Available indicators suggest that economic act...,statement,NaN,NaN,2025-12-10
415,2025-12-10,"December 9-10, 2025",https://www.federalreserve.gov/monetarypolicy/...,NaN,minutes,NaN,NaN,2025-12-10


In [34]:
event_windows_df.to_csv('../../data/processed/events_window.csv', index=False)
events.to_csv('../../data/processed/events_all.csv', index=False)
estimation_df.to_csv('../../data/processed/estimation_window.csv', index=False)